# 08 - 数据增强**学习目标：**- 理解数据增强在目标检测中的重要性- 掌握常见增强策略：几何变换、色彩变换、Mosaic、MixUp- 学会配置和调整增强参数- 对比增强前后的训练效果---

## 1. 为什么需要数据增强**问题**：标注数据很贵，但模型需要大量数据才能泛化。**解决**：通过对现有数据做变换，生成"新"的训练样本。```原始图片：                 增强后：┌──────────┐              ┌──────────┐│   🐱     │              │     🐱   │  ← 翻转│          │              │          │└──────────┘              └──────────┘┌──────────┐              ┌──────────┐│   🐱     │     →        │  🐱      │  ← 色彩变化│          │              │  (偏暗)   │└──────────┘              └──────────┘还可以：旋转、缩放、裁剪、模糊、噪声...```**关键**：增强只在训练时使用，推理时不使用！

## 2. YOLO 中的增强策略YOLO 内置了丰富的数据增强：| 增强类型 | 参数 | 默认值 | 说明 ||----------|------|--------|------|| **HSV-H** | hsv_h | 0.015 | 色调随机偏移 || **HSV-S** | hsv_s | 0.7 | 饱和度随机偏移 || **HSV-V** | hsv_v | 0.4 | 亮度随机偏移 || **旋转** | degrees | 0.0 | 随机旋转角度 || **平移** | translate | 0.1 | 随机平移比例 || **缩放** | scale | 0.5 | 随机缩放范围 || **翻转** | fliplr | 0.5 | 左右翻转概率 || **Mosaic** | mosaic | 1.0 | 4图拼接概率 || **MixUp** | mixup | 0.0 | 2图混合概率 |

## 3. Mosaic 增强详解**Mosaic 是 YOLOv4 引入的突破性增强**：```┌───────┬───────┐│ img1  │ img2  ││       │       │├───────┼───────┤  → 4 张图拼成 1 张│ img3  │ img4  ││       │       │└───────┴───────┘优点：1. 丰富上下文：一张图里出现多种物体2. 增加小目标数量：4 张图的目标都在一张图里3. 批次多样性：一张图覆盖更多场景```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path

# 从 COCO128 取 4 张图
img_dir = Path("../data/coco128/images/train2017")
imgs = sorted(img_dir.glob("*.jpg"))[:4]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img_path in zip(axes, imgs):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.suptitle("Mosaic 增强的 4 张输入图")
plt.tight_layout()
plt.show()

## 4. MixUp 增强**MixUp**：将两张图片按比例混合。```img1 × α  +  img2 × (1-α)  =  mixed_img标签也按同样比例混合：label1 × α  +  label2 × (1-α)```**作用**：增加正则化，防止过拟合。

## 5. 增强参数配置

In [ ]:
# 教学演示 — 下面的实现帮助理解原理，生产代码请用 yolo_learn.*
from ultralytics import YOLO# 默认增强配置（YOLO 已经有很好的默认值）model = YOLO("yolo11n.pt")results_default = model.train(    data="../configs/data/coco128.yaml",    epochs=10,    device="mps",    project="../outputs",    name="aug_default",    # 使用默认增强)

In [ ]:
# 关闭增强（对比实验）model_no_aug = YOLO("yolo11n.pt")results_no_aug = model_no_aug.train(    data="../configs/data/coco128.yaml",    epochs=10,    device="mps",    project="../outputs",    name="aug_none",        # 关闭所有增强    hsv_h=0,    hsv_s=0,    hsv_v=0,    degrees=0,    translate=0,    scale=0,    fliplr=0,    mosaic=0,    mixup=0,)

In [ ]:
# 生产路径：使用 yolo_learn 增强预设
from yolo_learn.data.augment import get_augment_params

# params = get_augment_params("medium")  # 可选: none, light, medium, heavy
# results = train(model="yolo11n.pt", data="...", **params)


In [ ]:
# 强化增强model_heavy = YOLO("yolo11n.pt")results_heavy = model_heavy.train(    data="../configs/data/coco128.yaml",    epochs=10,    device="mps",    project="../outputs",    name="aug_heavy",        # 强化增强    hsv_h=0.03,    hsv_s=0.9,    hsv_v=0.6,    degrees=15,    translate=0.2,    scale=0.8,    mosaic=1.0,    mixup=0.2,)

## 6. 对比不同增强策略

In [ ]:
import matplotlib.pyplot as pltimport pandas as pdfrom pathlib import Path# 对比训练结果configs = {    "默认增强": "../outputs/aug_default",    "无增强": "../outputs/aug_none",     "强增强": "../outputs/aug_heavy",}fig, axes = plt.subplots(1, 2, figsize=(14, 5))for name, path in configs.items():    csv = Path(path) / "results.csv"    if csv.exists():        df = pd.read_csv(csv, skipinitialspace=True)        axes[0].plot(df["epoch"], df["train/box_loss"], label=name)        axes[1].plot(df["epoch"], df["metrics/mAP50(B)"], label=name)axes[0].set_title("Training Loss")axes[0].set_xlabel("Epoch")axes[0].legend()axes[0].grid(True, alpha=0.3)axes[1].set_title("mAP@0.5")axes[1].set_xlabel("Epoch")axes[1].legend()axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()

## 7. 增强策略建议| 场景 | 建议 ||------|------|| **数据很少** | 强增强：mosaic=1, mixup=0.2, 大旋转 || **数据充足** | 默认增强即可 || **小目标多** | 保持 mosaic=1（增加小目标数量） || **过拟合严重** | 增强增强强度 || **欠拟合** | 减弱增强，让模型更容易学习 |---**上一节**: [07 - 评估指标](07_evaluation_metrics.ipynb)**下一节**: [09 - 自定义数据集](09_custom_dataset.ipynb) - 准备自己的数据